# 06 — Test Prediction

Run this only after the complete test Source 1/2/3 files are placed in `data/test/`.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, joblib, sys
from rapidfuzz import fuzz
ROOT=Path.cwd().resolve()
if not (ROOT/'data').exists() and (ROOT.parent/'data').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from preprocessing import prepare_source
TEST=ROOT/'data'/'test'
s1=prepare_source(pd.read_csv(TEST/'test_source1.tsv',sep='\t'))
s2=prepare_source(pd.read_csv(TEST/'test_source2.tsv',sep='\t'))
s3=prepare_source(pd.read_csv(TEST/'test_source3.tsv',sep='\t'))
model=joblib.load(ROOT/'models'/'entity_resolution_random_forest.joblib')
threshold=float((ROOT/'models'/'threshold.txt').read_text().strip())
print(s1.shape,s2.shape,s3.shape,'threshold=',threshold)

In [ ]:
for df in (s1,s2,s3):
    df['name_country_key']=df['name_norm']+'||'+df['country_norm']
    df['address_block']=df['address_norm'].str.replace(' ','',regex=False).str[:8]
    df['address_country_key']=df['address_block']+'||'+df['country_norm']

def block_pairs(a,b):
    left=a[['entity_id','name_country_key','address_country_key']].rename(columns={'entity_id':'source1_entity_id'})
    right=b[['entity_id','name_country_key','address_country_key']].rename(columns={'entity_id':'candidate_entity_id'})
    n=left.merge(right,on='name_country_key',how='inner')[['source1_entity_id','candidate_entity_id']]
    d=left.merge(right,on='address_country_key',how='inner')[['source1_entity_id','candidate_entity_id']]
    return pd.concat([n,d],ignore_index=True).drop_duplicates()
candidates=pd.concat([block_pairs(s1,s2),block_pairs(s1,s3)],ignore_index=True).drop_duplicates()
print('candidate pairs:',len(candidates))

In [ ]:
smap=s1.set_index('entity_id').to_dict('index'); all_candidates=pd.concat([s2,s3],ignore_index=True).set_index('entity_id').to_dict('index')
def make_features(r):
    a=smap[r.source1_entity_id]; b=all_candidates[r.candidate_entity_id]
    n1,n2=a['name_norm'],b['name_norm']; ad1,ad2=a['address_norm'],b['address_norm']
    return [fuzz.WRatio(n1,n2)/100,fuzz.token_set_ratio(n1,n2)/100,fuzz.WRatio(ad1,ad2)/100,fuzz.token_set_ratio(ad1,ad2)/100,int(bool(n1) and n1==n2),int(a['country_norm']==b['country_norm']),int(not n1 or not n2),int(not ad1 or not ad2),abs(len(n1)-len(n2)),abs(len(ad1)-len(ad2))]
X=pd.DataFrame([make_features(r) for r in candidates.itertuples(index=False)],columns=['name_similarity','name_token_similarity','address_similarity','address_token_similarity','name_exact','country_match','name_missing','address_missing','name_length_diff','address_length_diff'])
candidates['prob']=model.predict_proba(X)[:,1]

In [ ]:
# Final candidate file: every test S1 exactly once.
cand_out=(candidates.groupby('source1_entity_id')['candidate_entity_id'].apply(lambda x: ','.join(sorted(set(x)))).reindex(s1['entity_id'],fill_value='').rename('candidate_entity_ids').reset_index())
cand_out.to_csv(ROOT/'output'/'candidate_pairs.tsv',sep='\t',index=False)

# Final matching file: only candidates above the selected threshold.
pred=candidates[candidates['prob']>=threshold]
pred_map=pred.groupby('source1_entity_id')['candidate_entity_id'].apply(lambda x: ','.join(sorted(set(x)))).to_dict()
match_out=pd.DataFrame({'source1_entity_id':s1['entity_id'],'matched_entity_ids':[pred_map.get(x,'') for x in s1['entity_id']]})
match_out.to_csv(ROOT/'output'/'matching_results.tsv',sep='\t',index=False)
print(match_out.head())
print('Rows:',len(match_out),'non-empty:',(match_out.matched_entity_ids!='').sum())

In [ ]:
print('Final files:')
print(ROOT/'output'/'candidate_pairs.tsv')
print(ROOT/'output'/'matching_results.tsv')
print('Run src/submission_validator.py before submitting.')